[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_08_Productionizing_AI.ipynb)

# 🏭 Lesson 8: Productionizing AI
### *From prototype to production-grade AI systems*

---

**Where you are in the journey:**
- ✅ Lessons 1–7: You can build, chain, and retrieve with LLMs
- 🎯 **Today:** Make those systems production-worthy — observable, evaluatable, efficient, and resilient

**Why this lesson matters:**
> Building a working prototype is 20% of the job. The other 80% is making it reliable, measurable, and affordable at scale. Every AI engineer *must* understand this layer.

**What you'll learn today:**
1. **Observability** — Logging & tracing every LLM call so you can debug what went wrong
2. **Evals** — Measuring whether your AI is actually doing a good job
3. **Rate Limits & Retries** — Handling API limits gracefully without crashing
4. **Cost Optimization** — Counting tokens, caching, and choosing models wisely

---

In [ ]:
# 🚀 SETUP — Run this first (installs packages + loads your API key)
!pip install anthropic tiktoken tenacity -q

import anthropic
import json
import time
import logging
import hashlib
from datetime import datetime
from dataclasses import dataclass, field
from typing import Optional

# Load API key from Colab Secrets
# Steps: Left sidebar → 🔑 Secrets → Add secret named ANTHROPIC_API_KEY
try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
except ImportError:
    import os
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', 'your-key-here')

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print('✅ Setup complete!')

---
# Part 1: Observability — *"What did my LLM actually do?"*

## The Problem

In traditional software, when something breaks, you read the stack trace. With LLMs, the "bug" might be:
- A prompt that worked yesterday failing today
- Unexpectedly high latency
- A model returning nonsense for a specific input
- Cost spiraling because tokens are being wasted

Without observability, you're flying blind.

## What to Log on Every LLM Call

Think of each LLM call as an event with:
- **Inputs:** model, messages, temperature, max_tokens
- **Outputs:** response text, stop reason
- **Metrics:** input tokens, output tokens, latency, cost
- **Metadata:** timestamp, call ID, any user/session IDs

## The Pattern: Wrap Your LLM Client

Instead of calling `client.messages.create(...)` directly everywhere, you wrap it in an **instrumented function** that logs everything automatically.

In [ ]:
# --- LLM Trace Dataclass ---
# This is the structured record we create for EVERY LLM call

@dataclass
class LLMTrace:
    call_id: str
    model: str
    input_messages: list
    system_prompt: Optional[str]
    output_text: str
    input_tokens: int
    output_tokens: int
    total_tokens: int
    latency_ms: float
    stop_reason: str
    timestamp: str
    estimated_cost_usd: float

    def to_dict(self):
        return {
            'call_id': self.call_id,
            'timestamp': self.timestamp,
            'model': self.model,
            'input_tokens': self.input_tokens,
            'output_tokens': self.output_tokens,
            'total_tokens': self.total_tokens,
            'latency_ms': round(self.latency_ms, 1),
            'stop_reason': self.stop_reason,
            'estimated_cost_usd': round(self.estimated_cost_usd, 6),
            'output_preview': self.output_text[:100] + '...' if len(self.output_text) > 100 else self.output_text
        }

print('✅ LLMTrace dataclass defined')

In [ ]:
# --- The Instrumented LLM Client ---
# This wraps every API call and captures a full trace automatically

# Approximate pricing per million tokens (as of 2025)
PRICING = {
    'claude-haiku-4-5-20251001': {'input': 0.80, 'output': 4.00},   # $/million tokens
    'claude-sonnet-4-5':         {'input': 3.00, 'output': 15.00},
    'claude-opus-4-5':           {'input': 15.00, 'output': 75.00},
}

# In-memory trace store (in production: send to a database or logging service)
trace_log: list[LLMTrace] = []

def llm_call(
    messages: list,
    model: str = 'claude-haiku-4-5-20251001',
    system: str = None,
    max_tokens: int = 500,
    temperature: float = 0.7,
    label: str = 'unlabeled'
) -> tuple[str, LLMTrace]:
    """Instrumented LLM call: returns (text, trace). Logs everything automatically."""

    # Generate a unique call ID
    call_id = hashlib.md5(f"{label}{time.time()}".encode()).hexdigest()[:8]
    timestamp = datetime.utcnow().isoformat() + 'Z'

    # Build API params
    params = {
        'model': model,
        'max_tokens': max_tokens,
        'messages': messages,
    }
    if system:
        params['system'] = system

    # Time the actual API call
    t0 = time.perf_counter()
    response = client.messages.create(**params)
    latency_ms = (time.perf_counter() - t0) * 1000

    output_text = response.content[0].text
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens

    # Estimate cost
    pricing = PRICING.get(model, {'input': 3.0, 'output': 15.0})
    cost = (input_tokens / 1_000_000) * pricing['input'] + \
           (output_tokens / 1_000_000) * pricing['output']

    trace = LLMTrace(
        call_id=call_id,
        model=model,
        input_messages=messages,
        system_prompt=system,
        output_text=output_text,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        total_tokens=input_tokens + output_tokens,
        latency_ms=latency_ms,
        stop_reason=response.stop_reason,
        timestamp=timestamp,
        estimated_cost_usd=cost
    )

    trace_log.append(trace)

    print(f"[{label}] call_id={call_id} | {input_tokens}→{output_tokens} tokens | "
          f"{latency_ms:.0f}ms | ${cost:.6f}")

    return output_text, trace

print('✅ Instrumented llm_call() ready')

In [ ]:
# --- Test it! Make a few calls and watch the trace log ---

# Call 1: Simple question
text1, trace1 = llm_call(
    messages=[{'role': 'user', 'content': 'What is the capital of France? Answer in one word.'}],
    label='geography-question'
)

# Call 2: Slightly bigger prompt
text2, trace2 = llm_call(
    messages=[{'role': 'user', 'content': 'List 3 benefits of using agents over single LLM calls.'}],
    system='You are a helpful AI educator. Be concise.',
    label='agent-benefits'
)

# --- Now inspect the trace log ---
print('\n\n📊 Full Trace Log:')
print('=' * 60)
for trace in trace_log:
    print(json.dumps(trace.to_dict(), indent=2))
    print()

# --- Session summary ---
total_tokens = sum(t.total_tokens for t in trace_log)
total_cost = sum(t.estimated_cost_usd for t in trace_log)
avg_latency = sum(t.latency_ms for t in trace_log) / len(trace_log)
print(f'📈 Session Summary: {len(trace_log)} calls | {total_tokens} total tokens | \
${total_cost:.5f} total | {avg_latency:.0f}ms avg latency')

# 💡 EXPERIMENT: Try adding more calls and watch the session totals grow.
# 💡 EXPERIMENT: Change the model to 'claude-sonnet-4-5' and compare costs.

## 🔑 Key Observability Takeaways

- **Every call generates a trace** — you now have a record of exactly what was sent, what came back, and what it cost.
- **In production**, you'd ship traces to a system like **LangSmith**, **Helicone**, **Langfuse**, or even just your own database/CloudWatch.
- **Latency** is your first alert signal — if it spikes, something changed (model, prompt length, API issues).
- **Token counts don't lie** — if input tokens are growing unexpectedly, your context window is probably bloating.

---

# Part 2: Evals — *"Is my AI actually doing a good job?"*

## The Problem

How do you know if your AI is getting better or worse after you change a prompt? You need **evals** — automated tests for LLM behavior.

Evals are the equivalent of unit tests, but for AI outputs. Unlike traditional unit tests, AI outputs are fuzzy — the answer might be correct but worded differently. That's the challenge.

## Types of Evals

| Type | How It Works | Best For |
|------|-------------|----------|
| **Exact match** | `output == expected` | Structured outputs (JSON, numbers, single words) |
| **Contains check** | `expected in output` | Key facts that must appear |
| **LLM-as-judge** | Another LLM grades the output | Open-ended responses, quality assessment |
| **Human eval** | A human rates the output | Gold standard, but expensive/slow |

## The Eval Harness Pattern

A **harness** is a framework that:
1. Takes a dataset of `(input, expected_output)` pairs
2. Runs your system on each input
3. Grades each output with a scorer
4. Aggregates scores into metrics you can track over time

In [ ]:
# --- Define an Eval Dataset ---
# Each entry: (input, expected_output, eval_type)

EVAL_DATASET = [
    {
        'id': 'e001',
        'input': 'What is 2 + 2?',
        'expected': '4',
        'eval_type': 'contains'  # output must contain '4'
    },
    {
        'id': 'e002',
        'input': 'What is the capital of Japan?',
        'expected': 'Tokyo',
        'eval_type': 'contains'
    },
    {
        'id': 'e003',
        'input': 'Explain what an LLM is in exactly one sentence.',
        'expected': 'A clear, accurate one-sentence explanation of what a large language model is',
        'eval_type': 'llm_judge'  # LLM will grade this one
    },
    {
        'id': 'e004',
        'input': 'What is the boiling point of water in Celsius?',
        'expected': '100',
        'eval_type': 'contains'
    },
    {
        'id': 'e005',
        'input': 'Write a haiku about programming.',
        'expected': 'A haiku (5-7-5 syllable structure) about programming that is creative and coherent',
        'eval_type': 'llm_judge'
    },
]

print(f'📋 Eval dataset loaded: {len(EVAL_DATASET)} test cases')
print(f'   Contains-check evals: {sum(1 for e in EVAL_DATASET if e["eval_type"]=="contains")}')
print(f'   LLM-judge evals: {sum(1 for e in EVAL_DATASET if e["eval_type"]=="llm_judge")}')

In [ ]:
# --- Scorer Functions ---

def score_contains(output: str, expected: str) -> dict:
    """Pass if expected string appears in output (case-insensitive)."""
    passed = expected.lower() in output.lower()
    return {
        'passed': passed,
        'score': 1.0 if passed else 0.0,
        'reason': f'Expected "{expected}" in output'
    }

def score_llm_judge(output: str, expected_criteria: str) -> dict:
    """Use a judge LLM to rate the output against a rubric. Returns 0.0–1.0."""

    judge_prompt = f"""You are an AI evaluator. Grade the following output against the criteria.

CRITERIA: {expected_criteria}

OUTPUT TO GRADE:
{output}

Respond in this exact JSON format:
{{"score": <0.0 to 1.0>, "reason": "<one sentence explanation>"}}

Score 1.0 = perfectly meets criteria. Score 0.0 = completely fails criteria. Be strict but fair."""

    judge_response, _ = llm_call(
        messages=[{'role': 'user', 'content': judge_prompt}],
        model='claude-haiku-4-5-20251001',
        max_tokens=150,
        label='llm-judge'
    )

    try:
        # Extract JSON from response
        import re
        json_match = re.search(r'\{.*?\}', judge_response, re.DOTALL)
        result = json.loads(json_match.group())
        result['passed'] = result['score'] >= 0.7
        return result
    except Exception as e:
        return {'passed': False, 'score': 0.0, 'reason': f'Failed to parse judge response: {e}'}

print('✅ Scorers ready')

In [ ]:
# --- The Eval Harness ---
# The system under test: a simple LLM call with a brief system prompt

def system_under_test(user_input: str) -> str:
    """This is the AI system we're evaluating. Change this to test different prompts/models."""
    output, _ = llm_call(
        messages=[{'role': 'user', 'content': user_input}],
        system='You are a helpful, accurate assistant. Be concise.',
        model='claude-haiku-4-5-20251001',
        max_tokens=200,
        label='eval-run'
    )
    return output


def run_evals(dataset: list) -> list:
    """Run all evals and return results."""
    results = []

    for eval_case in dataset:
        print(f"\n🔬 Running eval {eval_case['id']}: {eval_case['input'][:50]}...")

        # Get the model's actual output
        actual_output = system_under_test(eval_case['input'])

        # Score it
        if eval_case['eval_type'] == 'contains':
            score_result = score_contains(actual_output, eval_case['expected'])
        elif eval_case['eval_type'] == 'llm_judge':
            score_result = score_llm_judge(actual_output, eval_case['expected'])
        else:
            score_result = {'passed': False, 'score': 0.0, 'reason': 'Unknown eval type'}

        result = {
            'eval_id': eval_case['id'],
            'input': eval_case['input'],
            'actual_output': actual_output,
            'eval_type': eval_case['eval_type'],
            **score_result
        }

        status = '✅ PASS' if score_result['passed'] else '❌ FAIL'
        print(f"   {status} | Score: {score_result['score']:.1f} | {score_result['reason']}")
        results.append(result)

    return results


# --- Run the full eval suite ---
print('🚀 Running eval suite...\n')
eval_results = run_evals(EVAL_DATASET)

# --- Summary report ---
passed = sum(1 for r in eval_results if r['passed'])
avg_score = sum(r['score'] for r in eval_results) / len(eval_results)
print(f'\n📊 Eval Summary: {passed}/{len(eval_results)} passed | Avg score: {avg_score:.2f}')

# 💡 EXPERIMENT: Change the system prompt in system_under_test() and re-run.
# 💡 EXPERIMENT: Add a new eval case to EVAL_DATASET with eval_type='llm_judge'.

## 🔑 Key Evals Takeaways

- **Evals = unit tests for AI.** Run them before and after every prompt change.
- **LLM-as-judge** is surprisingly good for open-ended quality evaluation — but use a fast, cheap model (Haiku) for judging.
- **Track scores over time.** If your score drops from 0.9 → 0.7, something regressed — find it before your users do.
- **Real eval frameworks** you'll use later: **LangSmith**, **PromptFoo**, **RAGAS** (for RAG evaluation).

---

# Part 3: Rate Limits & Retries — *"Handle the real world gracefully"*

## The Problem

LLM APIs have **rate limits** — caps on:
- **RPM** (requests per minute)
- **TPM** (tokens per minute)
- **RPD** (requests per day)

When you hit a rate limit, the API returns a `429 Too Many Requests` error. A naive agent that crashes on 429 is not production-ready.

## The Solution: Exponential Backoff with Jitter

**Exponential backoff** = when a request fails, wait a bit, then retry. Each retry waits *longer* than the last (exponentially).

**Jitter** = add randomness to the wait time, so multiple clients don't all retry at the exact same moment (avoiding synchronized "thundering herd" storms).

```
Attempt 1: wait 1s → retry
Attempt 2: wait 2s → retry   (exponential)
Attempt 3: wait 4s → retry
Attempt 4: wait 8s → retry
... up to max_retries
```

The `tenacity` library handles all of this for us elegantly.

In [ ]:
# --- Retry Logic with tenacity ---
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
    before_sleep_log,
    RetryError
)
import anthropic

# Setup logger so tenacity prints what it's doing
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('retry-demo')


@retry(
    retry=retry_if_exception_type(anthropic.RateLimitError),  # only retry on 429s
    wait=wait_exponential(multiplier=1, min=1, max=60),        # 1s, 2s, 4s, 8s... up to 60s
    stop=stop_after_attempt(5),                                # give up after 5 attempts
    before_sleep=before_sleep_log(logger, logging.WARNING),    # log each retry
)
def resilient_llm_call(messages: list, model: str = 'claude-haiku-4-5-20251001',
                        max_tokens: int = 300) -> str:
    """LLM call that automatically retries on rate limit errors."""
    response = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        messages=messages
    )
    return response.content[0].text


# --- Simulate what this looks like ---
print('📋 What the retry decorator does:')
print('  1. Calls the API')
print('  2. If RateLimitError → wait 1s → retry')
print('  3. If RateLimitError again → wait 2s → retry')
print('  4. If RateLimitError again → wait 4s → retry')
print('  5. After 5 failures → raise RetryError')
print('  Otherwise → return response normally\n')

# Make a real call to show it works normally
try:
    result = resilient_llm_call(
        messages=[{'role': 'user', 'content': 'Say hello in exactly 3 words.'}]
    )
    print(f'✅ Response: "{result}"')
except RetryError:
    print('❌ All retries exhausted — you would handle this gracefully in production')

# 💡 EXPERIMENT: Also retry on anthropic.APIStatusError for 5xx server errors.
# 💡 EXPERIMENT: Add a circuit breaker — after 10 failures in 1 minute, stop trying.

In [ ]:
# --- Simulating What a 429 Response Looks Like ---
# (We can't actually trigger one easily, but let's demonstrate the pattern)

import random

# Simulate a flaky API that fails 40% of the time
call_count = [0]

class FakeRateLimitError(Exception):
    pass

@retry(
    retry=retry_if_exception_type(FakeRateLimitError),
    wait=wait_exponential(multiplier=0.1, min=0.1, max=2),  # fast for demo
    stop=stop_after_attempt(6),
)
def flaky_api_call(task_name: str) -> str:
    call_count[0] += 1
    attempt = call_count[0]

    if random.random() < 0.5:  # 50% chance of failure
        print(f"  Attempt {attempt} → ❌ RateLimitError (simulated)")
        raise FakeRateLimitError("429: Rate limit exceeded")

    print(f"  Attempt {attempt} → ✅ Success!")
    return f"Result of '{task_name}'"

# Run simulation
print('🎭 Simulating a flaky API (50% failure rate):\n')
for task in ['task-A', 'task-B', 'task-C']:
    call_count[0] = 0
    print(f"Calling API for {task}:")
    try:
        result = flaky_api_call(task)
        print(f"  Final: {result}\n")
    except Exception as e:
        print(f"  Final: FAILED after all retries — {e}\n")

# 💡 KEY INSIGHT: Without retry logic, this would fail ~50% of the time.
# With retries, you recover automatically. This is what makes agents production-grade.

## 🔑 Key Rate Limit Takeaways

- **Always wrap LLM calls with retry logic** in production. Period.
- **Retry ONLY on retriable errors**: 429 (rate limit), 529 (overloaded). Don't retry 4xx auth errors — they'll never succeed.
- **Tenacity** is the go-to Python library. In Node.js, use `p-retry`.
- **Jitter matters at scale** — if 100 agents all hit a limit and retry at `t+2s`, they'll all fail again. Randomize.
- **Also implement a client-side rate limiter** to *avoid* hitting limits in the first place: `time.sleep(60/RPM)` between calls.

---

# Part 4: Cost Optimization — *"Don't burn money you don't have to"*

## The Problem

LLM costs add up fast. At scale:
- A 1M-token/day system on Sonnet costs ~$3/day input + $15/day output = **~$5,000/year**
- The same system on Haiku costs ~$0.80 + $4/day = **~$1,700/year**
- With smart caching, you might cut that by 50%+

## Four Key Levers

| Lever | What It Does | Typical Savings |
|-------|-------------|----------------|
| **Token counting** | Know your costs before they hit | Visibility |
| **Model selection** | Use Haiku for simple tasks, Sonnet for complex | 3–20x cost reduction |
| **Response caching** | Skip the API call for identical inputs | 50–80% reduction on repetitive workloads |
| **Prompt compression** | Remove unnecessary words from prompts | 10–30% token reduction |

In [ ]:
# --- Lever 1: Token Counting Before You Send ---
# Use tiktoken to estimate token count BEFORE making the API call
# (Anthropic uses a similar BPE tokenizer; tiktoken with cl100k_base is a close approximation)

import tiktoken

def estimate_tokens(text: str, model: str = 'cl100k_base') -> int:
    """Estimate token count for a string (approximate for Claude)."""
    enc = tiktoken.get_encoding(model)
    return len(enc.encode(text))

def estimate_cost(input_tokens: int, output_tokens_estimate: int,
                  model: str = 'claude-haiku-4-5-20251001') -> float:
    """Estimate cost in USD before making the call."""
    pricing = PRICING.get(model, {'input': 3.0, 'output': 15.0})
    return (input_tokens / 1_000_000) * pricing['input'] + \
           (output_tokens_estimate / 1_000_000) * pricing['output']

# --- Demo: Count tokens for different prompts ---
prompts = [
    ("Short prompt", "What is the capital of France?"),
    ("Medium prompt", "Explain the difference between supervised and unsupervised learning in machine learning, including examples of each and when you would choose one over the other."),
    ("Long prompt", "You are an expert software architect. I need you to design a complete microservices architecture for an e-commerce platform that handles 1 million requests per day. Include service decomposition, communication patterns, data storage strategy, caching layers, and deployment considerations. Be thorough and include trade-offs for each decision.")
]

print('📏 Token count estimates:\n')
for name, prompt in prompts:
    tokens = estimate_tokens(prompt)
    cost_haiku = estimate_cost(tokens, 200, 'claude-haiku-4-5-20251001')
    cost_sonnet = estimate_cost(tokens, 200, 'claude-sonnet-4-5')
    cost_opus = estimate_cost(tokens, 200, 'claude-opus-4-5')
    print(f'  {name}:')
    print(f'    Tokens: {tokens}')
    print(f'    Cost → Haiku: ${cost_haiku:.6f} | Sonnet: ${cost_sonnet:.6f} | Opus: ${cost_opus:.6f}')
    print()

# 💡 INSIGHT: Even a 100-token difference multiplied by 1M calls/day = significant money.

In [ ]:
# --- Lever 2: Model Routing ---
# Route simple tasks to cheap models, complex tasks to powerful models

def route_to_model(prompt: str) -> str:
    """
    Smart model router:
    - Short/simple prompts → Haiku (cheapest)
    - Medium prompts → Sonnet
    - Long/complex prompts → Opus (most capable)
    """
    token_count = estimate_tokens(prompt)

    # Simple heuristic based on prompt length
    # In production you might also classify by task type (factual, creative, analysis)
    if token_count < 50:
        model = 'claude-haiku-4-5-20251001'
        reason = 'short/simple (< 50 tokens)'
    elif token_count < 200:
        model = 'claude-sonnet-4-5'
        reason = 'medium complexity (50-200 tokens)'
    else:
        model = 'claude-opus-4-5'
        reason = 'complex/long (> 200 tokens)'

    return model, reason, token_count


# Demonstrate routing
test_prompts = [
    "What is 2+2?",
    "Summarize the key differences between REST and GraphQL APIs, including pros and cons of each.",
    "Design a production-grade architecture for a real-time collaborative document editing system similar to Google Docs. Include the database design, conflict resolution strategy, WebSocket management, offline support, and how you'd handle concurrent edits from thousands of simultaneous users."
]

print('🔀 Model Router Demo:\n')
for prompt in test_prompts:
    model, reason, tokens = route_to_model(prompt)
    print(f'  Prompt: "{prompt[:60]}..."' if len(prompt) > 60 else f'  Prompt: "{prompt}"')
    print(f'  → Tokens: {tokens} | Model: {model} | Reason: {reason}')
    print()

# 💡 EXPERIMENT: Classify by task type instead of just length.
# Example: factual questions → Haiku, code generation → Sonnet, legal analysis → Opus

In [ ]:
# --- Lever 3: Response Caching ---
# If the same prompt is sent twice, don't call the API twice. Return the cached result.

import hashlib

class LLMCache:
    """Simple in-memory cache for LLM responses."""

    def __init__(self):
        self._store = {}    # {hash: (response, timestamp)}
        self.hits = 0
        self.misses = 0
        self.tokens_saved = 0

    def _make_key(self, messages: list, model: str, system: str = None) -> str:
        """Create a deterministic key from the request parameters."""
        payload = json.dumps({'model': model, 'system': system, 'messages': messages},
                              sort_keys=True)
        return hashlib.sha256(payload.encode()).hexdigest()

    def get(self, messages, model, system=None):
        key = self._make_key(messages, model, system)
        if key in self._store:
            self.hits += 1
            return self._store[key]
        self.misses += 1
        return None

    def set(self, messages, model, response_text, tokens_used, system=None):
        key = self._make_key(messages, model, system)
        self._store[key] = response_text

    def stats(self):
        total = self.hits + self.misses
        hit_rate = (self.hits / total * 100) if total > 0 else 0
        return f'Hits: {self.hits} | Misses: {self.misses} | Hit rate: {hit_rate:.1f}%'


cache = LLMCache()

def cached_llm_call(messages, model='claude-haiku-4-5-20251001', system=None, max_tokens=200):
    """LLM call with caching. Cache hit = zero API calls, zero cost."""

    # Check cache first
    cached = cache.get(messages, model, system)
    if cached:
        print(f'⚡ CACHE HIT — returning cached response (saved API call!)')
        return cached

    # Cache miss — call the API
    print(f'🌐 CACHE MISS — calling API...')
    response, trace = llm_call(messages=messages, model=model, system=system,
                                max_tokens=max_tokens, label='cached-call')
    cache.set(messages, model, response, trace.total_tokens, system)
    return response


# --- Demo: Same prompt called 3 times ---
same_prompt = [{'role': 'user', 'content': 'What is the meaning of life? Answer in one sentence.'}]

print('📦 Cache Demo — calling the same prompt 3 times:\n')
for i in range(3):
    print(f'Call #{i+1}:')
    result = cached_llm_call(same_prompt)
    print(f'Response: "{result[:80]}..."\n' if len(result) > 80 else f'Response: "{result}"\n')

print(f'\n📊 Cache Stats: {cache.stats()}')
print('💡 Calls 2 and 3 cost $0.00 — the cache returned instantly!')

# 💡 EXPERIMENT: Add TTL (time-to-live) to the cache — expire entries after 1 hour.
# 💡 EXPERIMENT: Use Redis instead of an in-memory dict for multi-process caching.

In [ ]:
# --- Lever 4: Prompt Compression ---
# Remove unnecessary words from prompts to reduce token usage

import re

def compress_prompt(text: str) -> str:
    """
    Simple prompt compression:
    - Remove filler phrases
    - Remove excessive whitespace
    - Collapse multiple newlines
    
    In production, you might use an LLM to compress its own context!
    """
    filler_phrases = [
        r'please note that\s+',
        r'it is important to note that\s+',
        r'as mentioned previously,?\s+',
        r'in other words,?\s+',
        r'to summarize,?\s+',
        r'needless to say,?\s+',
        r'it goes without saying that\s+',
        r'basically,?\s+',
        r'essentially,?\s+',
    ]

    result = text
    for phrase in filler_phrases:
        result = re.sub(phrase, '', result, flags=re.IGNORECASE)

    # Collapse excessive whitespace
    result = re.sub(r'\n{3,}', '\n\n', result)
    result = re.sub(r' {2,}', ' ', result)

    return result.strip()


# --- Demo ---
verbose_prompt = """
Please note that you are an AI assistant. It is important to note that your task is to
help users with their questions. Needless to say, you should be helpful and accurate.
Essentially, what I need is for you to explain what an API is in simple terms.
It goes without saying that you should be clear and concise. In other words, 
don't be overly verbose. Basically, just give me a clear explanation.
"""

compressed_prompt = compress_prompt(verbose_prompt)

original_tokens = estimate_tokens(verbose_prompt)
compressed_tokens = estimate_tokens(compressed_prompt)
savings_pct = (1 - compressed_tokens / original_tokens) * 100

print('📝 Original Prompt:')
print(verbose_prompt)
print(f'  → {original_tokens} tokens\n')

print('✂️ Compressed Prompt:')
print(compressed_prompt)
print(f'  → {compressed_tokens} tokens\n')

print(f'💰 Savings: {original_tokens - compressed_tokens} tokens ({savings_pct:.1f}% reduction)')

# 💡 EXPERIMENT: Write a system prompt that asks the LLM to compress its own context window summary.

## 🔑 Key Cost Optimization Takeaways

- **Count tokens before sending** — know your costs before they hit your bill.
- **Model routing is the biggest lever** — using Haiku instead of Opus for simple tasks can save 10–20x.
- **Caching is often overlooked** — FAQ bots, customer service agents, and RAG queries often have repeat inputs. Cache them.
- **Compress prompts** — especially long system prompts that repeat every call. 100 tokens × 10M calls = 1B tokens of savings.

---

# Putting It All Together: A Production-Ready Agent

Let's combine everything into one function that shows what a production-grade LLM call looks like.

In [ ]:
# --- The Production-Grade LLM Call Stack ---
# Combines: observability + caching + retries + model routing

from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

prod_cache = LLMCache()
prod_traces = []


def production_llm_call(
    messages: list,
    system: str = None,
    max_tokens: int = 500,
    label: str = 'production'
) -> str:
    """
    Production-grade LLM call:
    1. Check cache → skip API call if hit
    2. Auto-route to cheapest viable model
    3. Retry with exponential backoff on rate limits
    4. Log full trace for observability
    """

    # Step 1: Check cache
    user_content = ' '.join(m.get('content', '') for m in messages if m.get('role') == 'user')
    model, reason, token_count = route_to_model(user_content)

    cached = prod_cache.get(messages, model, system)
    if cached:
        print(f'[{label}] ⚡ Cache HIT — model={model} (would have cost ~${estimate_cost(token_count, 100, model):.6f})')
        return cached

    # Step 2: Make the (retry-wrapped) API call with tracing
    @retry(
        retry=retry_if_exception_type(anthropic.RateLimitError),
        wait=wait_exponential(multiplier=1, min=1, max=60),
        stop=stop_after_attempt(4),
    )
    def _call_with_retry():
        return llm_call(
            messages=messages,
            model=model,
            system=system,
            max_tokens=max_tokens,
            label=label
        )

    response_text, trace = _call_with_retry()
    prod_traces.append(trace)

    # Step 3: Cache the result
    prod_cache.set(messages, model, response_text, trace.total_tokens, system)

    return response_text


# --- Run the production agent ---
print('🏭 Production Agent Demo\n')

queries = [
    "What is 15 * 7?",
    "Explain the difference between TCP and UDP protocols.",
    "What is 15 * 7?",  # Repeat — should hit cache!
]

for query in queries:
    print(f'\n❓ Query: "{query}"')
    result = production_llm_call(
        messages=[{'role': 'user', 'content': query}],
        system='You are a helpful assistant. Be brief.',
        label='prod-demo'
    )
    print(f'💬 Answer: {result[:120]}...' if len(result) > 120 else f'💬 Answer: {result}')

# Final stats
print(f'\n📊 Production Stats:')
print(f'   Cache: {prod_cache.stats()}')
total_cost = sum(t.estimated_cost_usd for t in prod_traces)
print(f'   Total API cost: ${total_cost:.6f}')
print(f'   Total API calls made: {len(prod_traces)}')

---
# 🎓 Lesson 8 Summary

You've learned the four pillars of production AI systems:

| Pillar | What You Built | Production Tool Equivalent |
|--------|---------------|---------------------------|
| **Observability** | `LLMTrace` dataclass + session summaries | LangSmith, Langfuse, Helicone |
| **Evals** | Eval harness with contains + LLM-judge | PromptFoo, RAGAS, custom pipelines |
| **Retries** | `tenacity` with exponential backoff | Built into most LLM SDKs |
| **Cost Optimization** | Token counting, model routing, caching, compression | Anthropic's Prompt Caching, custom routers |

## The Mental Model

Think of your LLM calls like database queries:
- **Observability** = slow query logs
- **Evals** = integration tests
- **Retries** = connection pool retry logic
- **Caching** = query result cache
- **Token counting** = query cost estimation

You already know these concepts from Java/backend engineering. You're just applying them to a new layer.

---

# 🔭 What's Next — Lesson 9: Capstone Project

You've now learned **everything** needed to build a real AI agent:
- LLM fundamentals (L1)
- Prompt engineering (L2)
- Tool use (L3)
- Agent loops (L4)
- Memory (L5)
- Multi-agent systems (L6)
- RAG (L7)
- Production patterns (L8)

**Lesson 9 is your Capstone** — we'll design and build an open-source AI agent project from scratch. Something you can put on GitHub to demonstrate your skills.

Think about what problem you want to solve. Some ideas:
- 🔍 A research agent that searches, reads, and summarizes papers
- 🐛 A code review agent that analyzes PRs and suggests improvements
- 📧 An email triage agent that categorizes and drafts responses
- 💻 A CLI tool that turns natural language into shell commands

**Your homework:** Think about which problem excites you most. We'll build it together in Lesson 9.

---
*Lesson 8 of 9 | Daily AI Learning Curriculum | Delivered by Claude*